In [ ]:
# Change settings here, then Run All (Kaggle: T4 x2 + Internet).
from pathlib import Path
import os

PUZZLE_ID_START = 1020
PUZZLE_ID_END = 1020
INDICES_URL = None  # e.g. https://raw.githubusercontent.com/chervov/Test01/master/list555_1.txt
RANDOM_PUZZLE = False  # pick one index from the list/range
BEAM_WIDTH = 1_048_576  # or "max"
MAX_STEPS = 140
NUM_GPUS = None  # all visible GPUs, or an explicit count

MODEL_ROOT = None  # None: download the default models; or a local folder
COMPETITION_ROOT = None  # None: default puzzles; or a folder containing test.csv
CHECKPOINT_PATH = "q555_f1_bell2k.pt"  # Transformer: relative to MODEL_ROOT, or absolute
RESMLP_CHECKPOINT_PATH = "q555_2k_BEST.pt"  # MLP
LAYOUT_PATH = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8  # MLP weight = 1 - this value

SOLUTION_MODE = "collect"  # first | collect
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2_000
TOUCH_BFS_RADIUS = 0
TOUCH_BFS_MAX_ENTRIES = 1_048_576
REFLECT_MODE = "off"  # off | after_original | only
REFLECT_SOURCE_CSV = None  # required for only; columns puzzle_id,solution

RUN_TIMEOUT_SECONDS = 12 * 3600
WORK_ROOT = Path(os.environ.get("MULTIGPUBEAMSEARCH_HOME", Path.home() / ".cache/multigpubeamsearch/cube555"))
os.environ["MULTIGPUBEAMSEARCH_HOME"] = str(WORK_ROOT)
CACHE_DIR = WORK_ROOT / "native_cache"
OUTPUT_ROOT = Path.cwd() / "cube555_runs"
PUBLISH_RESULTS = True
AUTHOR = "trydotatwo"  # put your name here when forking
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = 169  # update to the saved version you run


# Cube555 through CayleyPy
Settings above are editable. Installation is folded below. Every collected solution is replayed before saving or publication. Publication uses the existing public results service; no GitHub token is needed. Set PUBLISH_RESULTS=False for private runs. The current public Cube555 submission format uses Kaggle provenance and two GPUs.

In [ ]:
# First-time installation (also runs automatically with Run All).
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve("https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/e727ed0215e6cea36aec23bff53fa47d1b67f954/kaggle/cube555_github/setup.py", setup_dir + "/setup.py")
    run_path(setup_file)


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch import cube555
from multigpubeamsearch.publication import publish_result


In [ ]:
puzzle_ids = cube555.select_puzzles(PUZZLE_ID_START, PUZZLE_ID_END,
    indices_url=INDICES_URL, random_choice=RANDOM_PUZZLE)
puzzle = cube555.load(puzzle_ids[0], cache_dir=WORK_ROOT,
    model_root=MODEL_ROOT, competition_root=COMPETITION_ROOT)
graph = cp.CayleyGraph(puzzle.definition, device="cpu", random_seed=42)
predictor = puzzle.blend(graph, transformer_weight=TRANSFORMER_WEIGHT,
    transformer_checkpoint=CHECKPOINT_PATH, mlp_checkpoint=RESMLP_CHECKPOINT_PATH,
    layout_path=LAYOUT_PATH)
beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS, cache_dir=CACHE_DIR,
    timeout_seconds=RUN_TIMEOUT_SECONDS, calibration_pipeline_seconds=1800))


In [ ]:
with beam.NativeSession() as session:
    for puzzle_id in puzzle_ids:
        puzzle = cube555.load(puzzle_id, cache_dir=WORK_ROOT,
            model_root=MODEL_ROOT, competition_root=COMPETITION_ROOT)
        result = graph.beam_search(
            start_state=puzzle.start_state, predictor=predictor, backend="native",
            beam_width=BEAM_WIDTH, max_steps=MAX_STEPS, beam_mode="simple",
            use_child_scores=True, return_path=True,
            solution_mode=SOLUTION_MODE, collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS, bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=cube555.source_solution(REFLECT_SOURCE_CSV, puzzle_id, graph),
        )
        saved = cube555.save_result(result, graph, puzzle.start_state, OUTPUT_ROOT)
        print(puzzle_id, "solutions:", len(result.solutions), "best length:",
              result.path_length if result.path_found else None, "saved:", saved)
        if PUBLISH_RESULTS and result.path_found:
            notebook_path = Path("/kaggle/working/__notebook__.ipynb")
            if len(result.native_metadata["devices"]) != 2 or not notebook_path.is_file():
                print("Not published: cayleypy-beam-results v1 requires a two-GPU Kaggle run. "
                      "Replay-checked result is saved locally:", saved)
            else:
                publish_result(result, graph, puzzle.start_state, puzzle_id=puzzle_id,
                    predictor=predictor, author=AUTHOR, kaggle_owner=KAGGLE_OWNER,
                    kaggle_slug=KAGGLE_SLUG, kaggle_version=KAGGLE_VERSION,
                    notebook_path=str(notebook_path),
                    solver_commit=os.environ["MULTIGPUBEAMSEARCH_SOURCE_COMMIT"],
                    directory=OUTPUT_ROOT)
